# Parte 5 — Boas práticas de Python: type hints, try/except e logging

**Objetivo desta parte:** até aqui, construímos o pipeline inteiro (extração,
tratamento, agregação, persistência) com um estilo propositalmente simples —
`print()` para acompanhar o progresso, poucas anotações de tipo, nenhum tratamento
de erro de rede. Funcionou porque a Open-Meteo é estável e os notebooks rodam sob
supervisão direta (dá pra ver o erro e reagir na hora).

Um script que roda sozinho (agendado, em produção) não tem essa sorte. Esta parte
pega a função de requisição da [Parte 1](01_extracao_open_meteo.ipynb)
(`buscar_clima_historico`) e a evolui, camada por camada, até perto do que já
existe de verdade em
[`clima_pipeline/extract/open_meteo_client.py`](../src/clima_pipeline/extract/open_meteo_client.py)
— o objetivo é entender **por que** o pacote de produção é escrito daquele jeito,
não só copiar a sintaxe. Para a referência teórica completa dessas três práticas
(incluindo PEP 8, docstrings e debug no VSCode), veja
[`src/README.md`](../src/README.md).

In [1]:
import logging
import time
from dataclasses import dataclass

import requests

Reaproveitamos os mesmos parâmetros e a mesma cidade de teste da Parte 1.

In [2]:
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"
VARIAVEIS_HORARIAS = [
    "temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m",
]
DATA_INICIO = "2025-01-01"
DATA_FIM = "2025-01-31"

## 1. Type hints além do básico

Já usamos anotações simples (`lat: float`, `-> dict`) desde a Parte 1. Duas
ferramentas a mais que aparecem bastante em código real:

- **Tipos opcionais** com `T | None` — documenta na própria assinatura que a
  função pode não encontrar o que foi pedido. É o padrão de
  [`resolver_slug_cidade`](../src/clima_pipeline/config.py), que devolve
  `str | None` (`None` quando a cidade não é reconhecida).
- **`@dataclass`** para agrupar um resultado com múltiplos campos em vez de um
  dict solto ou uma tupla posicional — cada campo tem nome e tipo, e o editor
  autocompleta `resultado.erro` em vez de você lembrar se era `resultado[1]` ou
  `resultado[2]`.

In [3]:
@dataclass
class ResultadoBusca:
    cidade: str
    payload: dict | None
    erro: str | None

    @property
    def sucesso(self) -> bool:
        return self.erro is None

Vamos usar exatamente essa classe daqui a pouco para representar o resultado de
cada tentativa de busca — sucesso **ou** falha, nunca os dois, sempre no mesmo
formato para quem consome o resultado (nada de checar `isinstance(x, dict)` vs.
`isinstance(x, Exception)` espalhado pelo código).

## 2. `try`/`except`: tratando falha de rede como parte do fluxo

Relembrando a função original da Parte 1:

```python
def buscar_clima_historico(lat, lon, data_inicio, data_fim) -> dict:
    resposta = requests.get(BASE_URL, params=params, timeout=30)
    resposta.raise_for_status()
    return resposta.json()
```

Se a API estiver fora do ar, a internet cair, ou uma cidade específica demorar
demais para responder, `requests` lança uma exceção (`ConnectionError`, `Timeout`,
ou `HTTPError` via `raise_for_status()`) — todas subclasses de
[`requests.exceptions.RequestException`](https://requests.readthedocs.io/en/latest/api/#exceptions).
Sem `try`/`except`, essa exceção sobe direto e **mata o loop inteiro**: se a 3ª de
5 cidades falhar, perdemos também o resultado das duas primeiras (que já tínhamos
conseguido).

### Duas estratégias, dois lugares diferentes no pacote

O projeto usa `try`/`except` em **camadas diferentes, com objetivos diferentes**:

- [`OpenMeteoClient.fetch_historical`](../src/clima_pipeline/extract/open_meteo_client.py)
  **não** captura o erro de rede — deixa `raise_for_status()` propagar. Essa camada
  só sabe falar com a API; decidir o que fazer com uma falha é responsabilidade de
  quem chama.
- [`dashboard/app.py`](../src/clima_pipeline/dashboard/app.py) captura no **limite
  da aplicação**, onde existe alguém (o usuário) para quem mostrar uma mensagem em
  vez de uma stack trace:

```python
try:
    cidades_df = carregar_cidades(api_base_url)
except requests.RequestException as erro:
    st.error(f"Não foi possível falar com a API em '{api_base_url}': {erro}")
    st.stop()
```

Regra prática: **capture exceções específicas** (nunca um `except:` puro, que
também captura `KeyboardInterrupt`/`SystemExit` e esconde bugs de digitação como
`NameError`) e só onde você realmente sabe o que fazer com o erro.

### Um exemplo real (para fins didáticos)

Para ver a exceção acontecer de verdade sem depender da Open-Meteo estar fora do
ar, apontamos para um domínio sob o TLD `.invalid` — reservado pela
[RFC 2606](https://www.rfc-editor.org/rfc/rfc2606) especificamente para nunca
resolver, o que torna o teste determinístico.

In [4]:
URL_INEXISTENTE = "https://esta-api-nao-existe.open-meteo.invalid/v1/archive"

try:
    requests.get(URL_INEXISTENTE, timeout=5)
except requests.exceptions.ConnectionError as erro:
    print(f"Falha de conexão tratada: {type(erro).__name__}")
except requests.exceptions.RequestException as erro:
    print(f"Outro erro de requisição tratado: {type(erro).__name__}")

Falha de conexão tratada: ConnectionError


## 3. `logging`: substituindo `print` por níveis e configuração central

A versão acima ainda só nos diz o que aconteceu via `print` dentro do próprio
`except` — não dá para desligar seletivamente, filtrar por severidade, nem mandar
para um arquivo. Configuramos logging exatamente como
[`logging_config.py`](../src/clima_pipeline/logging_config.py) faz em produção,
simplificado para rodar direto na célula do notebook (sem arquivo rotativo).

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    force=True,
)
logger = logging.getLogger("clima_pipeline.notebooks.06")

`force=True` é necessário aqui porque o `ipykernel` (o que roda o Jupyter) já
configura um handler no logger raiz ao iniciar — sem isso, `logging.basicConfig`
seria **no-op** (só configura se ainda não existir nenhum handler), e a formatação
acima nunca apareceria. É a mesma preocupação por trás da flag `_CONFIGURADO` em
`logging_config.py`, só que para o problema oposto: lá o guard existe para evitar
reconfigurar (e duplicar handlers) quando `setup_logging()` é chamado mais de uma
vez no mesmo processo — por exemplo, se o pipeline e a API rodassem juntos em um
teste.

## 4. Juntando tudo: retry com backoff, tipado e logado

A versão final: tenta de novo automaticamente em caso de falha transitória (até um
limite de tentativas), espera um pouco mais a cada tentativa (*backoff
exponencial* — evita bater na API instantaneamente várias vezes seguidas) e
devolve sempre um `ResultadoBusca` em vez de lançar a exceção para quem chamou.

Essa última decisão é diferente da de `OpenMeteoClient` (que prefere deixar o erro
propagar) — e tudo bem: a escolha certa depende de quem consome a função. Aqui
imaginamos um script que processa várias cidades e não quer parar por causa de
uma só.

In [6]:
def buscar_clima_historico_robusto(
    lat: float,
    lon: float,
    data_inicio: str,
    data_fim: str,
    cidade: str,
    url_base: str = BASE_URL,
    max_tentativas: int = 3,
    base_espera: float = 2.0,
) -> ResultadoBusca:
    """Busca o histórico horário com retry e backoff exponencial.

    Nunca lança — devolve sempre um ResultadoBusca; quem chama decide o que fazer
    olhando `resultado.sucesso`.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": data_inicio,
        "end_date": data_fim,
        "hourly": ",".join(VARIAVEIS_HORARIAS),
        "timezone": "America/Sao_Paulo",
    }

    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = requests.get(url_base, params=params, timeout=30)
            resposta.raise_for_status()
        except requests.exceptions.RequestException as erro:
            if tentativa == max_tentativas:
                logger.error(
                    "Falha ao buscar %s após %d tentativa(s): %s",
                    cidade, tentativa, erro,
                )
                return ResultadoBusca(cidade=cidade, payload=None, erro=str(erro))

            espera = base_espera ** tentativa
            logger.warning(
                "Tentativa %d/%d falhou para %s (%s) — tentando de novo em %.0fs",
                tentativa, max_tentativas, cidade, type(erro).__name__, espera,
            )
            time.sleep(espera)
        else:
            logger.info("Busca de %s concluída na tentativa %d", cidade, tentativa)
            return ResultadoBusca(cidade=cidade, payload=resposta.json(), erro=None)

    # Inalcançável (o loop sempre retorna ou é substituído por exceção antes de sair
    # normalmente), mas o type checker não sabe disso — sem este `raise`, o mypy
    # reclamaria de "missing return statement".
    raise AssertionError("loop deveria ter retornado antes daqui")

Rodando para uma cidade real — sucesso já na primeira tentativa, log em `INFO`:

In [7]:
resultado_sp = buscar_clima_historico_robusto(
    lat=-23.5505,
    lon=-46.6333,
    data_inicio=DATA_INICIO,
    data_fim=DATA_FIM,
    cidade="sao_paulo",
)

resultado_sp.sucesso, list(resultado_sp.payload["hourly_units"].items())[:3]

2026-08-18 20:57:18,742 | INFO     | clima_pipeline.notebooks.06 | Busca de sao_paulo concluída na tentativa 1


(True,
 [('time', 'iso8601'),
  ('temperature_2m', '°C'),
  ('relative_humidity_2m', '%')])

Rodando contra o domínio `.invalid` de propósito — falha em todas as tentativas,
log em `WARNING` a cada retry e `ERROR` na desistência final:

In [8]:
resultado_falho = buscar_clima_historico_robusto(
    lat=-23.5505,
    lon=-46.6333,
    data_inicio=DATA_INICIO,
    data_fim=DATA_FIM,
    cidade="cidade_fantasma",
    url_base=URL_INEXISTENTE,
    max_tentativas=2,
)

resultado_falho.sucesso, resultado_falho.erro

2026-08-18 20:57:18,756 | WARNING  | clima_pipeline.notebooks.06 | Tentativa 1/2 falhou para cidade_fantasma (ConnectionError) — tentando de novo em 2s


2026-08-18 20:57:20,765 | ERROR    | clima_pipeline.notebooks.06 | Falha ao buscar cidade_fantasma após 2 tentativa(s): HTTPSConnectionPool(host='esta-api-nao-existe.open-meteo.invalid', port=443): Max retries exceeded with url: /v1/archive?latitude=-23.5505&longitude=-46.6333&start_date=2025-01-01&end_date=2025-01-31&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Cwind_speed_10m&timezone=America%2FSao_Paulo (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x104fb2fd0>: Failed to resolve 'esta-api-nao-existe.open-meteo.invalid' ([Errno 8] nodename nor servname provided, or not known)"))


(False,
 'HTTPSConnectionPool(host=\'esta-api-nao-existe.open-meteo.invalid\', port=443): Max retries exceeded with url: /v1/archive?latitude=-23.5505&longitude=-46.6333&start_date=2025-01-01&end_date=2025-01-31&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Cwind_speed_10m&timezone=America%2FSao_Paulo (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x104fb2fd0>: Failed to resolve \'esta-api-nao-existe.open-meteo.invalid\' ([Errno 8] nodename nor servname provided, or not known)"))')

## Resumo

| Prática | Usada para | Onde ver em produção |
|---|---|---|
| Type hints (`T \| None`, `@dataclass`) | Deixar o contrato da função explícito no editor, sem esperar rodar para descobrir que algo pode ser `None` | [`config.py`](../src/clima_pipeline/config.py), [`pipeline.py`](../src/clima_pipeline/pipeline.py) |
| `try`/`except` específico | Não deixar uma falha transitória (rede, timeout) derrubar o processo inteiro; decidir o que fazer com o erro na camada certa | [`extract/open_meteo_client.py`](../src/clima_pipeline/extract/open_meteo_client.py) (deixa propagar), [`dashboard/app.py`](../src/clima_pipeline/dashboard/app.py) (trata na borda) |
| `logging` com níveis | Saber o que aconteceu depois do fato, com severidade e timestamp, sem espalhar `print()` | [`logging_config.py`](../src/clima_pipeline/logging_config.py) |

## Para praticar

Sugestões de exercício (sem resolver aqui):

- Adaptar `buscar_clima_historico_robusto` para aceitar uma **lista** de cidades e
  continuar processando as demais mesmo se uma falhar — devolvendo uma lista de
  `ResultadoBusca`, sucesso e falha misturados.
- Acrescentar `logger.debug(...)` logando os parâmetros exatos da requisição —
  útil para depurar, mas barulhento demais para rodar sempre em `INFO`.
- Abrir [`extract/open_meteo_client.py`](../src/clima_pipeline/extract/open_meteo_client.py)
  e esboçar (em texto, sem editar o pacote) onde um retry como este encaixaria,
  mantendo a decisão de "deixar propagar" para quem chama.